# MarketLens — real-source snapshot research
Runs Qwen and hybrid RAG on a private, precollected public-source corpus. Collection happens locally under a free-tier search budget. No API keys are included in the notebook or dataset. This is a live smoke run, not a labeled quality benchmark.


In [ ]:
from pathlib import Path

DATA = next(p.parent for p in Path('/kaggle/input').rglob('sources.json'))
wheels = list(DATA.glob('*.whl'))
assert len(wheels) == 1


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheels[0]) + '[research]'], check=True)


In [ ]:
import importlib.metadata
import json
import time

import torch

from marketlens.collection import SnapshotSearch
from marketlens.models import ResearchRequest, Settings
from marketlens.pipeline import run_research

assert torch.cuda.is_available(), 'Enable GPU'
torch.manual_seed(42)
print('GPU:', torch.cuda.get_device_name(0), flush=True)
request = ResearchRequest.model_validate_json((DATA / 'request.json').read_text())
search = SnapshotSearch(DATA / 'sources.json')
settings = Settings(rounds=0)
started = time.monotonic()
bundle = run_research(request, settings, '/kaggle/working/marketlens', search=search)
runtime = {'gpu': torch.cuda.get_device_name(0), 'duration_seconds': time.monotonic() - started, 'versions': {name: importlib.metadata.version(name) for name in ['torch', 'transformers', 'sentence-transformers', 'langgraph', 'faiss-cpu', 'marketlens']}, 'status': bundle.status, 'claims': len(bundle.claims), 'evidence_chunks': len(bundle.evidence)}
Path('/kaggle/working/marketlens/runtime.json').write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2), flush=True)
print('Warnings:', bundle.warnings, flush=True)


In [ ]:
import shutil

shutil.make_archive('/kaggle/working/marketlens-bundle', 'zip', '/kaggle/working/marketlens')


## Review before publication
Download the report and inspect every cited claim. Quote/identifier checks are not entailment checks. Keep this run private until source support, pricing conditions and unknown cells have been reviewed.
